# Midpoint and Hamiltonian Euler — matched runs, then reviewed maximum batch

Use the same Tesla T4 and software as the completed baseline. Keep its Drive artifacts. This notebook clones the published GitHub repository; no source ZIP upload is needed. Run all cells to execute correctness, separate smokes, both exact 50M matched runs and the review proposal. The maximum phase requires the protocol planning checkpoint and stays disabled during Run All. Failed gates stop execution and preserve logs.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import hashlib, json, os, shutil, subprocess, sys, time, zipfile
from datetime import datetime, timezone
from pathlib import Path

REPO_URL = 'https://github.com/LokeshJatangi/Reversability.git'
REPO_BRANCH = 'main'
REPO = Path('/content/Reversability')
PERSIST_ROOT = Path('/content/drive/MyDrive/Reversability')
ARTIFACTS = PERSIST_ROOT / 'artifacts'
ARTIFACTS.mkdir(parents=True, exist_ok=True)
SESSION_LOG = ARTIFACTS / 'colab-session.log'
# GitHub is the default. A legacy ZIP in Drive must not override newer source.
USE_SOURCE_BUNDLE = False  # optional offline source handoff only
SOURCE_BUNDLE = PERSIST_ROOT / 'reversible-source.zip'
if USE_SOURCE_BUNDLE:
    if not SOURCE_BUNDLE.is_file(): raise FileNotFoundError(SOURCE_BUNDLE)
    REPO.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(SOURCE_BUNDLE) as archive:
        for name in archive.namelist():
            target = (REPO / name).resolve()
            if not target.is_relative_to(REPO.resolve()): raise RuntimeError('Invalid bundle path')
        archive.extractall(REPO)
else:
    if not (REPO / '.git').is_dir():
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
if not (REPO / 'scripts/run_reversible_study.py').is_file():
    raise RuntimeError('Reversible methods are missing from the source checkout. Verify that the implementation commit is published to GitHub main.')
if not (REPO / 'requirements.txt').is_file():
    raise FileNotFoundError(f'Cloned repository is incomplete: {REPO}')

def run_logged(command, log_path=SESSION_LOG, cwd=REPO):
    command = [str(item) for item in command]
    stamp = datetime.now(timezone.utc).isoformat()
    Path(log_path).parent.mkdir(parents=True, exist_ok=True)
    with Path(log_path).open('a', encoding='utf-8') as log:
        header = f'\n[{stamp}] $ ' + ' '.join(command) + '\n'
        print(header, end='')
        log.write(header); log.flush()
        process = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line); log.flush()
        code = process.wait()
        os.fsync(log.fileno())
    if code:
        raise subprocess.CalledProcessError(code, command)

print('Repository:', REPO)
print('Persistent artifacts:', ARTIFACTS)


In [ ]:
run_logged(['nvidia-smi'], ARTIFACTS / 'reversible-v1-environment.log')
run_logged([sys.executable, '-m', 'pip', 'install', '-q', '-r', REPO / 'requirements.txt'], ARTIFACTS / 'reversible-v1-environment.log')
run_logged([sys.executable, '-m', 'pytest', '-q'], ARTIFACTS / 'reversible-v1-environment.log')
run_logged([sys.executable, '-m', 'pip', 'freeze'], ARTIFACTS / 'reversible-v1-pip-freeze.log')


In [ ]:
# Locate exports elsewhere in MyDrive, or restore the verified recovery ZIP.
# If the baseline export is absent, upload baseline-reference-recovery.zip
# to MyDrive/Reversability once; extraction/verification below is automatic.
LOCAL_DATA = Path('/content/reversibility-data/data_fineweb_edu_gpt2_50m_v1')
DATA_RECEIPT = LOCAL_DATA.parent / 'baseline-data-receipt.json'
run_logged([sys.executable, '-u', REPO / 'scripts/stage_baseline_data.py',
            '--artifacts', ARTIFACTS, '--search-root', PERSIST_ROOT.parent,
            '--local-data', LOCAL_DATA, '--receipt', DATA_RECEIPT],
           ARTIFACTS / 'reversible-v1/data-staging.log')
data_receipt = json.loads(DATA_RECEIPT.read_text())
ARTIFACTS = Path(data_receipt['artifacts'])
print('Verified baseline artifacts:', ARTIFACTS)
print('Verified dataset source:', data_receipt['source_data'])


In [ ]:
run_logged([sys.executable, '-u', REPO / 'scripts/run_reversible_study.py', 'matched', '--artifacts', ARTIFACTS], ARTIFACTS / 'reversible-v1/matched-orchestration.log')


In [ ]:
run_logged([sys.executable, '-u', REPO / 'scripts/run_reversible_study.py', 'review', '--artifacts', ARTIFACTS], ARTIFACTS / 'reversible-v1/review-orchestration.log')


## Required planning checkpoint

Review correctness, losses/trajectories, memory, throughput, failures, remaining Colab budget and the proposed selection. Save the decision under `artifacts/reversible-v1/review-decision.json`. Only then set the flag below to true and run the maximum cell. No loss-qualified winner means report the loss and skip maximum training.


In [ ]:
# Deliberately false: Run All must stop for the required planning session.
RUN_MAXIMUM_AFTER_REVIEW = False
if RUN_MAXIMUM_AFTER_REVIEW:
    run_logged([sys.executable, '-u', REPO / 'scripts/run_reversible_study.py', 'maximum', '--artifacts', ARTIFACTS, '--max-batch', '256'], ARTIFACTS / 'reversible-v1/maximum-orchestration.log')
else:
    print('Matched runs/review proposal complete. Bring reversible-v1 artifacts back for the planning session before recording review-decision.json and enabling maximum phase.')
